# Body Composition Analysis
This notebook analyzes body composition changes based on fitness and lifestyle factors, predicting muscle definition and body fat changes.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict

# Load the data
try:
    df = pd.read_csv('fatLoss.csv')
    print('Dataset loaded successfully!')
except:
    print('Error loading dataset')
    exit()

print('Dataset Overview:')
print(df.head())
print('\nDataset Info:')
print(df.info())

## Calculate Basal Metabolic Rate (BMR)
Using the Mifflin-St Jeor Equation to calculate BMR based on gender, weight, height, and age.

In [ ]:
# Calculate BMR (Basal Metabolic Rate) using Mifflin-St Jeor Equation
df.loc[df['gender'] == 'M', 'BMR'] = (
    88.362
    + (13.397 * df.loc[df['gender'] == 'M', 'current_weight'])
    + (4.799 * df.loc[df['gender'] == 'M', 'height'])
    - (5.677 * df.loc[df['gender'] == 'M', 'age'])
)

df.loc[df['gender'] == 'F', 'BMR'] = (
    447.593
    + (9.247 * df.loc[df['gender'] == 'F', 'current_weight'])
    + (3.098 * df.loc[df['gender'] == 'F', 'height'])
    - (4.330 * df.loc[df['gender'] == 'F', 'age'])
)

## Feature Engineering
Calculate additional body composition metrics such as BMI, muscle mass index, and training intensity.

In [ ]:
# Generate a column 'BFP' with random percentages from 5 to 50%
np.random.seed(42)  # For reproducibility
df['BFP'] = np.random.uniform(5, 50, size=len(df))

In [ ]:
def calculate_body_composition_features(df):
    """Calculate additional features for body composition prediction"""

    # BMI (Body Mass Index)
    df['BMI'] = df['current_weight'] / ((df['height'] / 100) ** 2)

# IDEALLY ILL HAVE THE EXERCISE INTENSITY IN THE DATASET BUT FOR NOW ILL JUST USE 5

    # df['activity_level'] = df['frequency'] * df['exercise_intensity']
    df['activity_level'] = df['frequency'] * 5

    df['TDEE'] = df['BMR'] * df['activity_level']

    df['estimated_fat_mass'] = df['BFP'] * df['current_weight']

    # Estimate muscle mass based on training variables
    df['muscle_mass'] = (
        df['weight'] - df['estimated_fat_mass']
    )

    # Training intensity score
    df['training_intensity'] = (
        df['weight'] * df['sets'] * df['reps']) / df['current_weight']

    # Caloric efficiency
    df['caloric_efficiency'] = df['actual_fat_loss'] / \
        (df['daily_deficit'] / 100)

    # Sleep quality factor
    df['sleep_quality'] = np.where(df['sleep'] >= 7.5, 1, df['sleep'] / 7.5)

    # Body size indicator
    df['body_size_index'] = (df['current_weight'] * df['height']) / 10000

    return df


# Apply body composition calculations
df = calculate_body_composition_features(df)

In [ ]:
# Handle categorical data - create separate encoders for each categorical variable
le_gender = LabelEncoder()
le_experience = LabelEncoder()
le_type = LabelEncoder()

df['gender_encoded'] = le_gender.fit_transform(df['gender'])
df['experience_encoded'] = le_experience.fit_transform(df['experience'])
df['type_encoded'] = le_type.fit_transform(df['type'])

# Define features and target
features = [
    'age', 'gender_encoded', 'sets', 'reps', 'weight', 'frequency', 
    'protein', 'calories', 'sleep', 'experience_encoded', 
    'genetic_advantage', 'daily_deficit', 'current_weight', 'height', 'type_encoded'
]
target = 'actual_fat_loss'

# Check if all features exist in the dataframe
missing_features = [f for f in features if f not in df.columns]
if missing_features:
    print(f"Error: The following required features are missing from the dataset: {missing_features}")
else:
    X = df[features]
    y = df[target]
    
    # Split data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print("Data split successfully!")

In [ ]:
# Initialize and train the Random Forest Regressor model
rf_regressor = RandomForestRegressor(n_estimators=100, random_state=42)
rf_regressor.fit(X_train, y_train)

# Make predictions on the test set
y_pred = rf_regressor.predict(X_test)

# Evaluate the model
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"Model trained successfully!")
print(f"Mean Squared Error: {mse:.2f}")
print(f"R-squared: {r2:.2f}")

In [ ]:
def predict_fat_loss(user_data: dict):
    """
    Predicts weight loss and calculates an adjustment factor based on a trained model.

    Args:
        user_data (dict): A dictionary containing a new user's information.
    
    Returns:
        A dictionary with the calculated values.
    """
    
    # Create a DataFrame from user data
    user_df = pd.DataFrame([user_data])
    
    # Encode categorical data for the new user using the correct encoders
    user_df['gender_encoded'] = le_gender.transform(user_df['gender'])
    user_df['experience_encoded'] = le_experience.transform(user_df['experience'])
    user_df['type_encoded'] = le_type.transform(user_df['type'])
    
    # Select the same features used for training
    user_X = user_df[features]

    # Calculate Ideal Weight Loss (in kg) based on a 90-day period
    # 1 kg of fat is approximately 7700 kcal
    time_frame_days = 90  # Assuming a 90-day period for the calculation
    daily_deficit = user_data['daily_deficit']
    ideal_weight_loss_kg = (daily_deficit * time_frame_days) / 7700
    
    # Predict actual fat loss using the trained model
    predicted_fat_loss_kg = rf_regressor.predict(user_X)[0]
    
    # Calculate the adjustment factor
    if ideal_weight_loss_kg > 0:
        adjustment_factor = predicted_fat_loss_kg / ideal_weight_loss_kg
    else:
        adjustment_factor = 0 # No adjustment for no ideal weight loss
        
    return {
        "ideal_weight_loss_kg": ideal_weight_loss_kg,
        "predicted_fat_loss_kg": predicted_fat_loss_kg,
        "adjustment_factor": adjustment_factor
    }

# Example usage with a new user's data
new_user_info = {
    'age': 30,
    'gender': 'M',
    'sets': 4,
    'reps': 10,
    'weight': 80,
    'frequency': 3,
    'protein': 180,
    'calories': 2500,
    'sleep': 8.0,
    'experience': 'Intermediate',
    'genetic_advantage': 3,
    'daily_deficit': 500,
    'current_weight': 85,
    'height': 180,
    'type': 'Compound'
}

# Run the prediction
results = predict_fat_loss(new_user_info)

# Print the results
print("USING RANDOMLY GENERATED BFPS AND INTENSITY SET TO 5 THIS IS NOT ACCURATE")
print("\n--- Fat Loss Prediction Results ---")
print(f"Ideal Weight Loss (over 90 days): {results['ideal_weight_loss_kg']:.2f} kg")
print(f"Predicted Fat Loss (over 90 days): {results['predicted_fat_loss_kg']:.2f} kg")
print(f"Adjustment Factor: {results['adjustment_factor']:.2f}")

In [ ]:
# we have muscle mass and BFP - can use these to get definition (multiply and scaled to 1-10)
df['definition_score'] = (df['muscle_mass'] * df['BFP']) / 100
df['definition_score'] = df['definition_score'].clip(1, 10)



In [ ]:
# Generate BFP columns for 3, 6, 9, and 12 months with decreasing values over time
np.random.seed(42)  # For reproducibility

# Start with the initial BFP and decrease over time
df['bfp_at_3'] = df['BFP'] - np.random.uniform(1, 3, size=len(df))
df['bfp_at_6'] = df['bfp_at_3'] - np.random.uniform(1, 2, size=len(df))
df['bfp_at_9'] = df['bfp_at_6'] - np.random.uniform(0.5, 1.5, size=len(df))
df['bfp_at_12'] = df['bfp_at_9'] - np.random.uniform(0.5, 1.0, size=len(df))

# Calculate predicted weight after fat loss for each row
df['predicted_weight_after_loss'] = df['current_weight'] - df['actual_fat_loss']

# df['predicted_weight_after_loss'].head()

# Calculate new muscle mass index at 3, 6, 9, and 12 months
df['muscle_mass_at_3'] = df['predicted_weight_after_loss'] - (df['bfp_at_3'] * df['predicted_weight_after_loss'] / 100)
df['muscle_mass_at_6'] = df['predicted_weight_after_loss'] - (df['bfp_at_6'] * df['predicted_weight_after_loss'] / 100)
df['muscle_mass_at_9'] = df['predicted_weight_after_loss'] - (df['bfp_at_9'] * df['predicted_weight_after_loss'] / 100)
df['muscle_mass_at_12'] = df['predicted_weight_after_loss'] - (df['bfp_at_12'] * df['predicted_weight_after_loss'] / 100)

In [ ]:
df.columns

In [ ]:
# we have BFP at 3 months, 6, 9 and 12 in the dataset
# we can randomforestregressor it
# we'll get BFP drops using this
# and then we can use BFP drop to get definition score
# get new muscle mass
# new muscle mass at 3  = new weight - (df['bfp_at_3'] * df['predicted_weight_after_loss'])

# alright so we'll get BFP drops using randomforestregressor and then use those and the actualmusclemassdrop to get definition score

In [ ]:
# we got weight drop in kg
# in prediction we can get definition score by multiplying BFP  * current muscle mass

# so we need to predict new weight, new muscle mass, new BFP to get defintion

# we have muscle mass and BFP - can use these to get definition (multiply and scaled to 1-10)

